# 04C — Đọc baseline hoặc experiment 8 cấu hình theo tên

Read-only: không tạo features, train hoặc chạy lại model trên TEST.
Đặt `EXPERIMENT_NAME` bằng tên đã dùng trong 04D; để trống để xem baseline cố định.
Protocol v2 dùng validation Macro-F1 để chọn checkpoint; TEST tắt khi tuning.
Bảng, histories, confusion matrices và diagnostics đều lấy từ experiment đã chọn.
Shortlist chỉ dùng validation; source-level metrics là chỉ số phụ.


In [ ]:
# Each notebook kernel discovers its own runtime checkout.
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
_repo_hint = os.environ.get("EDGEAI_REPO_ROOT")
if _repo_hint and os.name != "nt" and PureWindowsPath(_repo_hint).drive:
    raise RuntimeError("EDGEAI_REPO_ROOT must name the checkout in this runtime, not a Windows drive")
_start = Path(_repo_hint).expanduser().resolve() if _repo_hint else Path.cwd().resolve()
_candidates = [_start, *_start.parents, Path("/content/edge-ai")]
_repo = next((p for p in _candidates if (p / "tools/project_paths.py").is_file()), None)
if _repo is None:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout and rerun this cell") from None
    _repo = Path(_repo_hint or "/content/edge-ai").expanduser().resolve()
    if _repo.exists():
        raise RuntimeError(f"{_repo} exists but is not an EdgeAI checkout; set EDGEAI_REPO_ROOT to the correct checkout")
    subprocess.run(["git", "clone", "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git", str(_repo)], check=True)
if not (_repo / "tools/notebook_runtime.py").is_file():
    raise RuntimeError("Checkout is missing notebook_runtime.py. Pull the latest GitHub commit in this runtime, then rerun this cell")
sys.path.insert(0, str(_repo / "tools"))
from notebook_runtime import initialize
PATHS = initialize(_repo)
from project_paths import REPO_ROOT, ProjectPaths, resolve_path, logical_relative
from pathlib import Path
import sys
sys.dont_write_bytecode = True
PROJECT = REPO_ROOT
sys.path.insert(0, str(PROJECT / "tools"))
import importlib
import stage04_data, model_zoo
for module in (stage04_data, model_zoo):
    importlib.reload(module)
import pandas as pd
from IPython.display import display
from model_zoo import FAMILIES, architecture_config, build_model
from stage04_data import BRANCHES, CLASSES, ROOT, prepare_caches, read_json


In [ ]:
# Để trống: baseline cố định. Nhập tên đã dùng trong 04D: kết quả experiment trên Drive.
EXPERIMENT_NAME = "exp_004_train_norm"

all_complete = False  # Prevent plots from reusing a previous selection after a failed check.
RUNS = None
result_root = (PATHS.experiment(EXPERIMENT_NAME) / "results"
               if EXPERIMENT_NAME else PATHS.BASELINE_ROOT)
print("Đang đọc:", result_root)
status = []
for frontend in BRANCHES:
    for family in FAMILIES:
        path = result_root / "runs" / f"{frontend}_{family}_seed42" / "result.json"
        status.append({"Frontend": frontend, "Model": family,
                       "Status": read_json(path)["status"] if path.is_file() else "NOT RUN"})
display(pd.DataFrame(status))
if EXPERIMENT_NAME:
    metadata = read_json(result_root.parent / "config/metadata.json")
    print("Experiment status:", metadata["status"],
          f'{metadata["completed_runs"]}/{metadata["expected_runs"]}')
    display(pd.Series({"experiment_id": metadata["experiment_id"],
                       "git_commit": metadata["git_commit"],
                       "description": metadata["configuration"]["description"],
                       "configuration": metadata["configuration"],
                       "runtime": metadata["runtime"]}, name="Run provenance"))
    from experiment_review import verify_experiment, comparison_summary
    records = verify_experiment(EXPERIMENT_NAME)
    if len(records) != 8:
        raise RuntimeError("04C requires a complete eight-configuration experiment; use the full-suite template in 04D.")
    summary, comparison = comparison_summary(records)
else:
    from baseline_review import verify_baseline
    import experiment_training
    records = verify_baseline()
    comparison = pd.DataFrame([experiment_training.comparison_row(r) for r in records])
    summary = read_json(result_root / "comparison.json")
RUNS = result_root / "runs"
all_complete = True
print("Saved training protocol/parameters:", read_json(result_root / "protocol.json")["training_configuration"])
print("TEST evaluated:", all(r.get("test_evaluated", True) for r in records))
display(comparison)
print("Provisional Stage-05 shortlist (validation only):", summary["stage05_shortlist"])


## Saved training histories and confusion matrices

TRAIN history là online metrics trong lúc cập nhật trọng số; validation chạy eval.
V2 bổ sung TRAIN eval tại checkpoint đã chọn để so sánh công bằng hơn.
Đường đánh dấu thể hiện epoch chọn theo F1 và epoch tốt nhất theo loss.
Confusion matrices có hàng là nhãn thật, cột là nhãn dự đoán; luôn giữ đủ bốn lớp.
V2 hiển thị validation và TRAIN eval; TEST chỉ hiện nếu experiment thực sự đã đánh giá TEST.


In [ ]:
if all_complete:
    import matplotlib.pyplot as plt
    labels_short = ["Ara", "Fun", "Squ", "Cpx"]
    for frontend in BRANCHES:
        for family in FAMILIES:
            directory = RUNS / f"{frontend}_{family}_seed42"
            result = read_json(directory / "result.json")
            history = read_json(directory / "history.json")
            fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
            epochs = [h["epoch"] for h in history]
            for split in ("train", "validation"):
                axes[0].plot(epochs, [h[split]["loss"] for h in history], label=split)
                axes[1].plot(epochs, [h[split]["macro_f1"] for h in history], label=split)
            if "train_eval_metrics" in result:
                axes[0].scatter([result["best_epoch"]], [result["train_eval_metrics"]["loss"]], label="TRAIN eval checkpoint", marker="x")
                axes[1].scatter([result["best_epoch"]], [result["train_eval_metrics"]["macro_f1"]], label="TRAIN eval checkpoint", marker="x")
            for ax in axes[:2]:
                ax.axvline(result["best_epoch"], color="green", linestyle="--", label="selected epoch")
                if "checkpoint_candidates" in result:
                    ax.axvline(result["checkpoint_candidates"]["minimum_loss"]["epoch"], color="gray", linestyle=":", label="min-loss epoch")
                ax.legend(fontsize=7)
            axes[0].set(title="Weighted loss", xlabel="Epoch")
            axes[1].set(title="Macro-F1", xlabel="Epoch")
            secondary = result.get("final_test_metrics") or result.get("train_eval_metrics")
            secondary_title = "TEST" if result.get("final_test_metrics") else "TRAIN eval"
            for ax, metrics, title in ((axes[2], result["selected_validation_metrics"], "Validation"),
                                       (axes[3], secondary, secondary_title)):
                if metrics is None:
                    ax.set_visible(False)
                    continue
                cm = metrics["confusion_matrix"]
                ax.imshow(cm, cmap="Blues")
                for i in range(4):
                    for j in range(4):
                        ax.text(j, i, cm[i][j], ha="center", va="center")
                ax.set(title=title + " window confusion", xlabel="Predicted", ylabel="True",
                       xticks=range(4), yticks=range(4), xticklabels=labels_short, yticklabels=labels_short)
            fig.suptitle(f"{EXPERIMENT_NAME or 'fixed baseline'} / {frontend} / {family}; selected epoch {result['best_epoch']}")
            fig.tight_layout(); plt.show()
            print("Validation per-class:")
            display(pd.DataFrame(result["selected_validation_metrics"]["per_class"]).T)
            if "validation_source_metrics" in result:
                display(pd.Series(result["validation_source_metrics"], name="Validation source evaluation"))
            if result.get("final_test_metrics") is not None:
                print("TEST per-class (descriptive, not for tuning):")
                display(pd.DataFrame(result["final_test_metrics"]["per_class"]).T)
                display(pd.Series(result["source_test_metrics"], name="TEST source evaluation"))
            else:
                print("TEST chưa được đánh giá trong experiment này; dùng validation để tuning.")
            if (directory / "diagnostics.json").is_file():
                diagnostics = read_json(directory / "diagnostics.json")
                print("Validation species × acquisition method (support + recall):")
                display(pd.DataFrame(diagnostics["validation_subgroups"]))
                print("BatchNorm running statistics tại checkpoint được chọn:")
                display(pd.DataFrame(diagnostics["batchnorm_selected_checkpoint"]).T)


## So sánh validation với experiment đối chứng

Đọc reference từ config đã lưu, hoặc nhập tên khác ở cell dưới.
So sánh ghép cặp cùng model/frontend; không dùng TEST để chọn cấu hình.
Kiểm tra cột Same runtime và criterion trước khi diễn giải chênh lệch.


In [ ]:
REFERENCE_EXPERIMENT_NAME = ""  # Để trống: dùng reference_experiment_id đã lưu trong config.
if all_complete and EXPERIMENT_NAME:
    from experiment_review import compare_validation_to_reference
    reference_name = REFERENCE_EXPERIMENT_NAME or metadata["configuration"].get("reference_experiment_id")
    if reference_name:
        reference_path = PATHS.experiment(reference_name)
        if (reference_path / "config/metadata.json").is_file():
            display(compare_validation_to_reference(EXPERIMENT_NAME, reference_name))
        else:
            print("Chưa tìm thấy reference:", reference_path, "; kết quả hiện tại vẫn được xác thực riêng.")
    else:
        print("Experiment cũ không có reference; nhập REFERENCE_EXPERIMENT_NAME nếu muốn so sánh.")


## Limits before Stage 05

No Edge winner is declared. Float32 checkpoint size is not INT8 size, RAM, Flash or latency.
This one-seed comparison makes no claims of superiority to published literature. Multiple windows may
share one recording; source-name isolation does not prove biological/session/domain independence.
The frozen acquisition-method confounding remains. Read [Stage 04 methodology](../../docs/STAGE04.md).
